[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module7/10-diffusion-models.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module7/10-diffusion-models.ipynb)

# Diffusion Models
**Module 7 — Lesson 10 | Estimated time: 50 minutes**

> 💡 Enable GPU: Runtime → Change runtime type → GPU (required — image generation needs GPU)

## Learning Objectives
By the end of this notebook you will be able to:
- Explain the forward (noise addition) and reverse (denoising) processes of diffusion
- Visualise the DDPM noise schedule
- Generate images from text prompts using `StableDiffusionPipeline`
- Apply negative prompts and tune inference steps
- Understand ControlNet for conditioned generation
- Use the inpainting and img2img pipelines

In [ ]:
!pip install -q diffusers accelerate transformers

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: Stable Diffusion requires a GPU. Please enable GPU in Runtime settings.')
dtype = torch.float16 if device == 'cuda' else torch.float32

## 1. Diffusion Process Intuition

**Forward process** (fixed, no learning): gradually add Gaussian noise to a clean image over `T` steps:
$$q(x_t | x_{t-1}) = \mathcal{N}(x_t;\, \sqrt{1-\beta_t}\, x_{t-1},\, \beta_t I)$$

**Reverse process** (learned): a neural network (usually a U-Net) learns to denoise `x_t` back to `x_{t-1}`:
$$p_\theta(x_{t-1} | x_t) = \mathcal{N}(x_{t-1};\, \mu_\theta(x_t, t),\, \Sigma_\theta)$$

At inference, we start from pure Gaussian noise `x_T ~ N(0, I)` and iteratively denoise.

In [ ]:
# Visualise the DDPM forward process (adding noise)
T = 1000
betas = np.linspace(1e-4, 0.02, T)           # linear schedule
alphas = 1 - betas
alpha_bars = np.cumprod(alphas)

# Create a simple gradient image
img = np.zeros((64, 64, 3))
img[:, :32] = [0.2, 0.5, 0.9]   # left: blue
img[:, 32:] = [0.9, 0.3, 0.2]   # right: red

timesteps_to_show = [0, 100, 250, 500, 750, 999]
fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(14, 2.5))
for ax, t in zip(axes, timesteps_to_show):
    noise = np.random.randn(*img.shape)
    ab    = alpha_bars[t]
    noisy = np.sqrt(ab) * img + np.sqrt(1 - ab) * noise
    ax.imshow(np.clip(noisy, 0, 1))
    ax.set_title(f't={t}', fontsize=10)
    ax.axis('off')
plt.suptitle('DDPM Forward Process: Adding Noise Over Time', fontsize=12)
plt.tight_layout(); plt.show()

# Noise schedule
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
axes[0].plot(betas);       axes[0].set_title('β_t (noise schedule)'); axes[0].set_xlabel('t')
axes[1].plot(alphas);      axes[1].set_title('α_t = 1 - β_t');       axes[1].set_xlabel('t')
axes[2].plot(alpha_bars);  axes[2].set_title('ᾱ_t = Π α_s');         axes[2].set_xlabel('t')
plt.tight_layout(); plt.show()

## 2. Text-to-Image with Stable Diffusion

We use `stabilityai/stable-diffusion-2-1` (a commonly available SD checkpoint). On Colab T4 GPU, a 20-step generation takes about 5–10 seconds.

In [ ]:
from diffusers import StableDiffusionPipeline

# Load the pipeline (will download ~5GB on first run; cached afterwards)
pipe = StableDiffusionPipeline.from_pretrained(
    'stabilityai/stable-diffusion-2-1-base',
    torch_dtype=dtype,
    safety_checker=None,
)
pipe = pipe.to(device)
pipe.enable_attention_slicing()  # reduces VRAM usage
print('Pipeline loaded.')

In [ ]:
# Text-to-image generation
prompts = [
    'a golden retriever sitting in a field of sunflowers, oil painting, detailed',
    'futuristic city at night with neon lights, cyberpunk, photorealistic',
]

fig, axes = plt.subplots(1, len(prompts), figsize=(12, 5))
for ax, prompt in zip(axes, prompts):
    generator = torch.Generator(device=device).manual_seed(42)
    image = pipe(
        prompt,
        num_inference_steps=25,
        guidance_scale=7.5,
        generator=generator,
    ).images[0]
    ax.imshow(image)
    ax.set_title(textwrap.fill(prompt, 35), fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
import textwrap

# Negative prompts — guide the model away from unwanted features
prompt   = 'portrait of a wizard, fantasy art, detailed, dramatic lighting'
neg_prompt = 'blurry, low quality, distorted, ugly, watermark, text'

generator = torch.Generator(device=device).manual_seed(7)
image_neg = pipe(
    prompt,
    negative_prompt=neg_prompt,
    num_inference_steps=25,
    guidance_scale=8.5,
    generator=generator,
).images[0]

generator = torch.Generator(device=device).manual_seed(7)
image_no_neg = pipe(
    prompt,
    num_inference_steps=25,
    guidance_scale=8.5,
    generator=generator,
).images[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image_no_neg); axes[0].set_title('Without negative prompt'); axes[0].axis('off')
axes[1].imshow(image_neg);    axes[1].set_title('With negative prompt');    axes[1].axis('off')
plt.tight_layout(); plt.show()

## 3. Inference Steps Trade-off

In [ ]:
import time

steps_to_try = [5, 10, 20, 40]
prompt_test  = 'a red apple on a wooden table, photorealistic'

fig, axes = plt.subplots(1, len(steps_to_try), figsize=(14, 4))
for ax, steps in zip(axes, steps_to_try):
    gen = torch.Generator(device=device).manual_seed(42)
    t0  = time.time()
    img = pipe(prompt_test, num_inference_steps=steps,
               guidance_scale=7.5, generator=gen).images[0]
    elapsed = time.time() - t0
    ax.imshow(img)
    ax.set_title(f'{steps} steps\n{elapsed:.1f}s', fontsize=10)
    ax.axis('off')
plt.suptitle('Inference Steps Trade-off: Quality vs Speed', fontsize=12)
plt.tight_layout(); plt.show()

## 4. img2img Pipeline

img2img conditions the reverse diffusion on an existing image, allowing style transfer and variations.

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline

img2img_pipe = StableDiffusionImg2ImgPipeline(**pipe.components)
img2img_pipe = img2img_pipe.to(device)

# Use the apple image from the previous step as input
init_image = image_neg.resize((512, 512))

generator = torch.Generator(device=device).manual_seed(77)
output_img = img2img_pipe(
    prompt='a wizard casting a spell, fantasy digital art, vibrant',
    image=init_image,
    strength=0.65,          # 0 = no change, 1 = full diffusion
    guidance_scale=7.5,
    num_inference_steps=25,
    generator=generator,
).images[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(init_image);  axes[0].set_title('Input image');      axes[0].axis('off')
axes[1].imshow(output_img);  axes[1].set_title('img2img output');   axes[1].axis('off')
plt.tight_layout(); plt.show()

## 5. ControlNet — Concept and Usage

ControlNet adds spatial conditioning (edges, depth, pose, etc.) on top of an existing diffusion model. A trained ControlNet takes a conditioning image alongside the text prompt to guide generation.

**Architecture:** ControlNet copies the encoder layers of the U-Net and trains only those copies. The outputs are added to the original U-Net's decoder features via zero-convolution layers.

**Common ControlNet types:**
- `canny` — Canny edge conditioning
- `depth` — Depth map conditioning
- `pose` — OpenPose skeleton conditioning
- `scribble` — Hand-drawn scribble conditioning

In [ ]:
# Canny edge extraction for ControlNet demo
try:
    import cv2
    HAS_CV2 = True
except ImportError:
    HAS_CV2 = False
    print('cv2 not available — install with: !pip install opencv-python-headless')

if HAS_CV2:
    img_array = np.array(init_image.convert('RGB'))
    edges = cv2.Canny(img_array, threshold1=100, threshold2=200)
    edges_rgb = np.stack([edges]*3, axis=-1)  # make 3-channel
    edge_img  = Image.fromarray(edges_rgb)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(init_image); axes[0].set_title('Original');     axes[0].axis('off')
    axes[1].imshow(edge_img);   axes[1].set_title('Canny Edges');  axes[1].axis('off')
    plt.suptitle('ControlNet Input: Canny Edge Map', fontsize=12)
    plt.tight_layout(); plt.show()
    print('ControlNet would use these edges to guide image generation.')
    print('Usage: ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny")')
else:
    print('Demo requires opencv. The code structure remains the same.')

# ControlNet pipeline usage structure (requires extra VRAM — shown as code, not run)
controlnet_code = '''
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline
from diffusers.utils import load_image
import torch

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16
)
pipe_ctrl = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16,
).to("cuda")

image = pipe_ctrl(
    prompt="a detailed portrait of a person",
    image=edge_img,          # canny edge map
    num_inference_steps=20,
    controlnet_conditioning_scale=0.8,
).images[0]
'''
print('\nControlNet pipeline code:')
print(controlnet_code)

## Practice Exercises

**Exercise 1 — Guidance Scale Sweep**
Generate the same prompt with `guidance_scale` values of 1, 4, 7.5, 12, and 20. Display them in a row and describe how the guidance scale affects image quality, adherence to prompt, and artefacts.

**Exercise 2 — Inpainting**
Load a test image and create a binary mask (white region = area to repaint). Use `StableDiffusionInpaintPipeline` with the model `runwayml/stable-diffusion-inpainting` to repaint only the masked region with a new text prompt. Display original, mask, and result side by side.

**Exercise 3 — Prompt Interpolation**
Generate images for two opposite prompts (e.g. "summer sunny beach" and "winter snowy mountain"). Linearly interpolate the text embeddings using `pipe.text_encoder` and generate 5 images along the interpolation path. Display them as a visual transition.